In [1]:
import os 

os.chdir('/mnt/c/Users/harsh/OneDrive/Documents/course/thesis/testing_hardwicke/test_3')
os.getcwd()
from config import OPENAI_API_KEY, ANTHROPIC_API_KEY

In [2]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("API key not found.")
else:
    print("API key found.")

API key found.


In [3]:
api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    print("API key not found.")
else:
    print("API key found.")

API key found.


In [4]:
from anthropic import Anthropic
from openai import OpenAI
import pypdf

client = OpenAI()
checker = Anthropic()

/home/killmach/miniconda3/envs/thesis_API/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [5]:
import pypdf
# Function to extract text from specific pages of a PDF
def extract_pdf_text(pdf_path, start_page, end_page):
    with open(pdf_path, "rb") as f:
        reader = pypdf.PdfReader(f)
        extracted_text = ""
        # Note: page indices start at 0
        for i in range(start_page-1, end_page):
            extracted_text += reader.pages[i].extract_text() + "\n"
    return extracted_text

In [6]:
# Provide path for the main article
article_path = 'article.pdf'

#Select the start and end page from the article you require
start_page = 2
end_page = 4

# Page extractor
pdf_text = extract_pdf_text(article_path, start_page, end_page)

print(pdf_text)

326 Farooqui, Manly
2010). Although the predictive information in these stud-
ies was derived from consciously perceived sources, it is possible that similar information derived from subliminal aspects of task context could also be used. Cognition would certainly be much more efficacious if it made use of control-relevant information present in aspects of the task environment that were not consciously perceived.
Our study links two streams of research: One investi-
gates cognitive processes that can be elicited through subliminal cues (van Gaal & Lamme, 2012), and the other investigates implicit instantiation of control through con-trol-relevant statistical relations in the task context (Bugg & Crump, 2012). Our interest in this topic arose from studies outlined earlier in which a different masked prime was linked to each of the specific tasks between which participants had to switch (e.g., Lau & Passingham, 2007; Manly et al., 2014). Our starting point was trying to deter -
mine wheth

##### Loading the require datasets
    -Have to edit dataset to have 1 column (can be tried to fix)

In [8]:

#----------------------------------------------------------------------------------------------------------------#
# Loading and processing the dataset - depends on how data is structured and what format it is in.
# This section is highly dependent on the dataset format and structure.
# + 
# Specific data cleaning and processing steps for the dataset
#----------------------------------------------------------------------------------------------------------------#
import pandas as pd
import glob
import numpy as np

data_path = '/mnt/c/Users/harsh/OneDrive/Documents/course/thesis/testing_hardwicke/test_3/data/Experiment 1'

excel_files = glob.glob(os.path.join(data_path, '*.xls*'))

data_dict = {}

for file in excel_files:
    df = pd.read_excel(file, header = 0)
    df = df.iloc[0:250]
    data_dict[file] = df

dataset = pd.concat([data_dict[file] for file in data_dict.keys()], axis=0, ignore_index=True)

otm_map = {'O': 2, 'T': 4, 'M': 8}
dataset['Prime'] = dataset['Prime'].replace(otm_map).astype(int)

dataset['Participant'] = (dataset.index // 250) + 1

dataset.to_csv('dataset.csv', index=False)

######### If codebook for the dataset is available, load it here
codebook_path = 'Codebook.xls'
codebook = pd.read_excel(codebook_path, header=None, sheet_name='Sheet1')
codebook_str = codebook.to_string()
print(codebook)

# Save the dataset as loaded to ensure final code works well.
#dataset.to_csv('dataset.csv', index=False)

                0                                           1  \
0        TaskType                                           1   
1             NaN                                           2   
2      TrialType                                            1   
3             NaN                                           2   
4   Primes (Cues)                                           O   
5             NaN                                           M   
6             NaN                                           T   
7        RespCorr                                           1   
8             NaN                                       False   
9          stay_2  Repeat trials following Non-Predictive Cue   
10         stay_4      Repeat trials following Predictive Cue   
11          swt_2  Switch trials following Non-Predictive Cue   
12          swt_8      Switch trials following Predictive Cue   
13           lnum                         Number on Left side   
14           rnum        

/tmp/ipykernel_31867/2240248548.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset['Prime'] = dataset['Prime'].replace(otm_map).astype(int)


In [9]:
#----------------------------------------------------------------------------------------------------------------#
# Creating a snippet of the dataset for LLM processing
# +
# Add any additional context regarding dataset and compile into a message for LLM processing
#----------------------------------------------------------------------------------------------------------------#
from data_snippet import make_llm_snippets

dataset_preview, dataset_summary, cat_col = make_llm_snippets(
    file_path = 'dataset.csv',
    id_cols = ['Participant'],
    extra_strata = [],
    n_per_stratum = 10
)

add_dataset_context = """
The dataset provided is a sample from the main dataset used in the experiment.
It contains 10 random observations from each of the 21 participants. Each participant in actuality has 250 observations.
Regarding some off the columns - namely: 'swt_2.1', 'swt_8.1', 'stay_2.1' & 'stay_4.1', these are all boolean types coded as 0 - False and 1 - True.
The codebook is provided in the doc_id = dataset_codebook, which has details regarding the most of the variables / columns in the dataset. The one's not mentioned in the codebook can be assumeded to be self-explanatory.
"""

In [10]:
print(dataset_preview)
print(dataset_summary)

### Dataset sample – 10 rows per stratum (Participant)

|   Block_Number |   Event_Number |   Prime |   PrimeVisible |   TaskType |   TrialType |   CorrResp |       RT |   RespCorr |   lnum |   rnum |   lFont |      swt |     stay |   stay_2 |   stay_4 |    swt_2 |    swt_8 |   swt_2.1 |   swt_8.1 |   stay_2.1 |   stay_4.1 |   Participant |
|----------------|----------------|---------|----------------|------------|-------------|------------|----------|------------|--------|--------|---------|----------|----------|----------|----------|----------|----------|-----------|-----------|------------|------------|---------------|
|              4 |            138 |       2 |              1 |          2 |           2 |          1 |  497     |          1 |     96 |     99 |       1 |  497     |  nan     |  nan     |  nan     |  497     |  nan     |         1 |       nan |        nan |        nan |             1 |
|              4 |             53 |       2 |              1 |          2 |        

In [8]:
import base64

# select total number of images / figre:
tot_img = 3

# Provide path for plots / figure / images
#   Format for providing path
#   path_1 = <path_of_fig_1>
#   path_2 = <path_of_fig_2>
#   ....

path_1 = 'images/fig_2.png'
path_2 = 'images/fig_s1.png'
path_3 = 'images/supMat_1.png'

########## Automated code starts here ##########

path_list_fig = []

for i in range(tot_img):
    path_list_fig.append(globals()[f'path_{i+1}'])

# Function to encode image to base64
def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


encode_list_fig = {}

for i, path in enumerate(path_list_fig):
    encode_list_fig[f'figure_{i+1}'] = encode_image(path)

FileNotFoundError: [Errno 2] No such file or directory: 'images/fig_2.png'

Function to create model response for each figure

In [11]:
def create_model_response(encode_figure):
    response_figure = client.chat.completions.create(
        model = 'gpt-4o',
        messages = [
            {
                'role': 'user',
                'content': [
                    {"type": "text", 'text': "Provide a detailed description of the graph / figure provided in the image which can be easily understood by another Large language model, allowing it to reimagine the whole figure. The other LLM can not see the image, so the description should be very detailed."},
                    {
                        "type": "image_url",
                        "image_url": {
                            'url': f"data:image/png;base64,{encode_figure}"
                        }
                    }
                ]
            }
        ]
    )

    return response_figure.choices[0].message.content

# Create a dictionary of model responses for each figure:
model_responses = {}
for figure_name, encode_data in encode_list_fig.items():
    model_responses[figure_name] = create_model_response(encode_data)

In [12]:
print(model_responses['figure_2'])

KeyError: 'figure_2'

In [13]:
import base64
# select total number of tables:
tot_tab = 1

# Provide path for plots / figure / images
#   Format for providing path
#   path_1 = <path_of_fig_1>
#   path_2 = <path_of_fig_2>
#   ....

path_1_tab = 'figure/table_2.png'
#path_2_tab = 'images/fig_s1.png'
#path_3_tab = 'images/supMat_1.png'

########## Automated code starts here ##########

path_list_tables = []

for i in range(tot_tab):
    path_list_tables.append(globals()[f'path_{i+1}_tab'])

# Function to encode image to base64
def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


encode_list_tables = {}

for i, path in enumerate(path_list_tables):
    encode_list_tables[f'table_{i+1}'] = encode_image(path)

In [14]:
## Table Extraction - not needed here
def create_table_response(table_image):
    response_table = client.chat.completions.create(
        model = 'gpt-4o',
        messages = [
            {
                'role': 'user',
                'content': [
                    {"type": "text", 'text': "Extract the table from the image and convert to a csv format, readable by a Large Language Model clearly."},
                    {
                        "type": "image_url",
                        "image_url": {
                            'url': f"data:image/png;base64,{table_image}"
                        }
                    }
                ]
            }
        ]
    )
    return response_table.choices[0].message.content

# Create a dictionary of model responses for each table:
model_responses_table = {}
for table_name, encode_data in encode_list_tables.items():
    model_responses_table[table_name] = create_table_response(encode_data)


In [15]:
print(model_responses_table['table_1'])

Sure, here is the table in CSV format:

```
Experiment,Primes Accuracy,Primes t test,Targets Accuracy
Experiment 1,"53.3 [50.2, 56.4]","t(16) = 2.06, p = .06","92.3 [89.8, 94.8]"
Experiment 2,"51.6 [49.4, 53.8]","t(21) = 1.47, p = .16","95.5 [93.3, 97.7]"
Experiment 3,"47.2 [44.8, 49.6]","t(20) = -2.27, p = .035","94.5 [92.1, 96.9]"
Experiment 4,"50.0 [48.8, 51.2]","t(21) = 0.04, p = .97","93.3 [91.3, 95.3]"
```

This format includes the experiment number, the primes accuracy with confidence intervals, the t-test statistics, and the targets accuracy with confidence intervals.


In [11]:
# Optimized prompt for the first stage:

dev_job = ('You are a reproducibility editor for a scientific journal, well trained in statistical analysis, your goal is to replicate the code for the analysis ' # you are a reproducibility editor for a scientific jounral, your sole goal is to reprordice the analysis of the article using the provided excerpt to you
            'as per the excerpt from provided article. The main motivation behind the research is not important to you, only the analysis '
            'process is important. '
            'Focus on the target results asked for when creating the analysis process. Do not worry about the full article.')

# ---------- PIPELINE INSTRUCTIONS (generic) ---------- #

pipeline_inst = ('Pipeline should contain a step by step process of exactly how the complete analysis process is suggested in the article. This pipleine should consider the following: \n ' #refer article_ref instead of article, additionally put a cut-off for any extra analysis suggested by article or model basis the reference
                 '1. import_libraries – list required packages  '
                 '2. read_data – load the datasets '
                 '3. clean_and_recode – handle missing data, type casting, relabeling variables per article '
                 '4. isolate_target_subset – keep rows/columns relevant to the focal analysis (conditions/tasks) '
                 '5. descriptive_summary – compute the summary measures as asked for in Reference excerpt (means, frequencies, etc.) '
                 '6. verify_assumptions – run only the assumption checks explicitly mentioned in the article '
                 '7. run_primary_analysis – execute the statistical test(s) the authors describe with their stated parameters '
                 '8. calculate_reported_effect_size – derive the effect-size metric the authors report (e.g., Cohen’s d, η², odds ratio) '
                 '9. compile_results – gather the exact figures the paper presents for tables/text.'
                 'Make sure to only have 1 set of numbering for the steps, and do not repeat any step. '
                 'Do not use any placeholders like "e.g." or "etc." in the steps, every step should be a concrete R function. ')

# Specific instructions for what part of analysis is to be reproduced.
hw_outcome_specs = ('For this article you should focus on the findings reported in the results section of Experiment 1. Specifically, you should attempt to reproduce all descriptive and inferential analyses reported in the text below and associated tables/figures:\n')
article_ref = (f"""
> Performance on switch trials, relative to repeat trials,
incurred a switch cost that was evident in longer RTs (836
vs. 689 ms) and lower accuracy rates (79% vs. 92%). If
participants were able to learn the predictive value of the
cue that preceded only switch trials and could instantiate
relevant anticipatory control in response to it, the performance
on switch trials preceded by this cue would be
better than on switch trials preceded by the nonpredictive
cue. This was indeed the case (mean RT—predictive
cue: 819 ms; nonpredictive cue: 871 ms; mean difference
= 52 ms, 95% confidence interval, or CI = [19.5,
84.4]), two-tailed paired t(20) = 3.34, p < .01. However,
error rates did not differ across these two groups of switch trials (predictive cue: 78.9%; nonpredictive cue: 78.8%), p = .8.
""")

# --------------------------------------------------------------------
# 1.  Function-calling schema  (machine-readable pipeline output)
# --------------------------------------------------------------------
tools = [
    {
        "type": "function",
        "function": {
            "name": "store_pipeline",
            "description": "Save the ordered list of analysis-pipeline steps",
            "parameters": {
                "type": "object",
                "properties": {
                    "steps": {
                        "type": "array",
                        "description": "Numbered analysis steps from data import to result compilation",
                        "items": {"type": "string"}
                    }
                },
                "required": ["steps"]
            }
        }
    }
]

# --------------------------------------------------------------------
# 2.  Prompt—using SYSTEM, DEVELOPER, ASSISTANT, USER roles
# --------------------------------------------------------------------
messages_client = [
    # ----- SYSTEM: overall persona & scope -----
    {
        "role": "system",
        "content": (
            f"{dev_job}  Focus exclusively on reproducing the statistical workflow "
            "described in the paper; ignore theoretical discussion or motivation."
        )
    },

    # ----- DEVELOPER: guard-rails & output contract -----
    {
        "role": "developer",
        "content": (
            "- Use only variables that appear in the provided dataset header.\n"
            "- Do not invent extra analyses, figures, or variable names.\n"
            "- Follow the numbered-step template exactly (see pipeline instructions).\n"
            "- When you are ready, CALL the `store_pipeline` function with the final list "
            "of steps; do not print the steps in plain text.\n"
            "- Make sure to identify column names by reading the header's of the dataset preview; "
            "those names must appear verbatim in your pipeline steps. Do make consideration of multi-indexing when applicable\n"
            "- Do not use placeholders like 'e.g.' or 'etc.'; every step must name a "
            "concrete R function. Also provide brief instructions with each step. "                # added step instructions here
        )
    },

    # ----- ASSISTANT context blocks (large, one-time inputs) -----
    {"role": "assistant", "name": "article_excerpt",        "content": pdf_text},
    #
    #{"role": "assistant", "name": "table_1_description",   "content": model_responses_table['table_1']},
    #
    {"role": "assistant", "name": "dataset_preview",        "content": dataset_preview},
    #
    {"role": "assistant", "name": "dataset_summary",    "content": dataset_summary},
    #
    {"role": "assistant", "name": "dataset_codebook",    "content": codebook_str},
    #
    {"role": "assistant", "name": "additional_dataset_context", "content": add_dataset_context},

    # ----- USER: task + focal outcome specs -----
    {
        "role": "user",
        "content": (
            # which results to focus on 
            f"{hw_outcome_specs}\n\n"
            "Reference excerpt:\n"
            f"{article_ref}\n\n"
            # pipeline instructions
            "### Pipeline instructions\n"
            f"{pipeline_inst}\n\n"
            "### Action\n"
            ######### Add any further instructions here specific to the article #########

            # Generic instruction for prodcing pipeline
            "If you understand, produce the pipeline by calling the `store_pipeline` "
            "function with the ordered list of steps."
        )
    }
]

In [12]:
# --------------------------------------------------------------------
# Chat completion request  (o3-mini model)
# --------------------------------------------------------------------
response = client.chat.completions.create(
    model  = "o3-mini",
    tools  = tools,
    messages = messages_client
)

In [14]:
########## Stub for the dataset and the article #############
for m in messages_client:
    if m.get('name') in {"article_excerpt", "dataset_preview", "additional_dataset_context"}:
       m['content'] = f"(See {m['name']} provided earlier; doc_id = {m['name']})"

# Print the API response for review
import json

assistant_reply_1 = response.choices[0].message.tool_calls[0]
tool_call_args = json.loads(assistant_reply_1.function.arguments)
pipeline_steps = tool_call_args['steps']
print("Pipeline steps:\n")
#for i, step in enumerate(pipeline_steps, start = 1):
#    print(f"{i}. {step}\n")

pipeline_str = "\n".join(f"{step}"                                                  # use "{idx}. {step}" if steps don't print as numbered list
                         for idx, step in enumerate(pipeline_steps, start=1))

print(pipeline_str)

pipeline_msg = {"role": "assistant", "content": pipeline_str}

assistant_tool_msg_1 = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": assistant_reply_1.id,
            "type": "function",
            "function": {
                "name": assistant_reply_1.function.name,
                "arguments": json.dumps(tool_call_args)
            }
        }
    ]
}

tool_response_msg_1 = {
    "role": "tool",
    "tool_call_id": assistant_reply_1.id,
    "content": "OK",
}

messages_client.extend([assistant_tool_msg_1, tool_response_msg_1])

Pipeline steps:

1. import_libraries: Load required packages with library(readr), library(dplyr), library(broom), and library(effsize).
2. read_data: Read the dataset from the CSV file using read.csv; for example, data <- read.csv('data.csv').
3. clean_and_recode: Convert columns to proper types with as.numeric and as.factor (e.g., data$RT <- as.numeric(data$RT), data$TrialType <- as.factor(data$TrialType)); recode binary columns (swt_2.1, swt_8.1) as factors if needed.
4. isolate_target_subset: Filter the dataset for switch trials by using filter(data, TrialType == 2) and then create two subsets: one for predictive switch trials using filter(., swt_8.1 == 1) and one for nonpredictive switch trials using filter(., swt_2.1 == 1).
5. descriptive_summary: For each Participant and each cue condition (predictive and nonpredictive), calculate the median RT with median(RT, na.rm = TRUE) and compute accuracy as the mean of CorrResp; then compute overall means across participants.
6. verify_ass

In [16]:
with open('Metrics_checker.txt', 'r') as file:
    metrics_check = file.read()

article_ref_wrapped = (
    "<<TARGET_RESULTS_START>>\n"
    f"{article_ref}\n"
    "<<TARGET_RESULTS_END>>"
)

# Codebook has been added here as it was available, can be removed if not available.
dataset_context = f"""
### Dataset preview and summary
{dataset_preview}

{dataset_summary}

{add_dataset_context}

{add_dataset_context}

### Dataset codebook

{codebook_str}
"""

# ---------------------------------------------------------------------
# 1  System prompt  (persona + hard guard-rails)
# ---------------------------------------------------------------------
system_prompt = (
    "You are a senior data-science reviewer. Your sole task is to evaluate and "
    "improve an analysis pipeline for a scientific article. \n\n"
    "Allowed scope\n"
    "-------------\n"
    "Focus strictly on logical soundness, completeness, and coding feasibility. "
    "**only for the results inside the tags <<TARGET_RESULTS_START>> ... <<TARGET_RESULTS_END>>**\n"
    "Ignore language style and theoretical interpretation. \n\n"

    "Output format\n"
    "-------------\n"
    "Return your evaluation **as a JSON object** with two keys:\n"
    " - \"metric_summary\": a list of {\"metric\", \"score\", \"comment\"}\n"
    " - \"revised_pipeline\": a numbered list (array of strings) that fixes\n"
    "    any weaknesses you identified.\n"
    "Scores range 0-100.  Comment only when improvement is needed.  Do **not**\n"
    "output any text outside the JSON object."
)

# ---------------------------------------------------------------------
# 2  Build the user message (all context + instructions + pipeline)
# ---------------------------------------------------------------------
user_prompt = f"""
### Goal 
{hw_outcome_specs}

### Focus tags with the results to reproduce
{article_ref_wrapped}

### Metrics for judging a pipeline
{metrics_check}

---

### Pipeline to review
{pipeline_str}

---

### Your tasks
1. Internally create your own pipeline (do **not** reveal it) to understand the
   target analysis.
2. Critique the reviewer’s pipeline using the supplied metrics (you may add
   well-defined metrics of your own, but explain them briefly in each comment).
3. Output **only** a JSON object with:
   - "metric_summary" – array of metric/score/comment triples  
   - "revised_pipeline" – the improved numbered pipeline

Remember: do not discuss numerical correctness of results, do not nit-pick
writing style, and do not output anything outside the JSON object.
"""
###### left out of the prompt for now ##########
### Figure 1 description (context only)
#{model_responses['figure_1']}
################################################




messages_checker = [
    # Context-only block (role=user or assistant both allowed)
    {"role": "user", "content": f"### Refer below for the excerpt from main article: \n{pdf_text}"},

    # Dataset preview (optional second context block)
    {"role": "user", "content": dataset_context},

    # Dataset codebook (optional second context block)
    {"role": "user", "content": codebook_str},

    # Table description (context-only block)
    #{"role": "user", "content": table_context},

    # Actual instruction block
    {"role": "user", "content": user_prompt}
]


In [17]:
response_h = checker.messages.create(
    model = 'claude-sonnet-4-20250514',
    system = f"{system_prompt}",
    messages = messages_checker,
    max_tokens = 4000
)



In [18]:
print(response_h.content[0].text)
checker_pipe = response_h.content[0].text

```json
{
  "metric_summary": [
    {
      "metric": "Specification Completeness (SC)",
      "score": 60,
      "comment": "Missing critical data filtering steps (Blocks 3&4, specific prime-cue mappings), unclear variable transformations, and incomplete specification of RT aggregation method (median vs mean confusion)"
    },
    {
      "metric": "Modularity Index (MI)", 
      "score": 75,
      "comment": "Generally good separation but step 4 compounds filtering operations and step 9 mixes result extraction with formatting"
    },
    {
      "metric": "Parameter Explicitness Ratio (PER)",
      "score": 40,
      "comment": "Missing key parameters: block selection criteria, prime-to-cue condition mapping, participant count (n=21), and specific statistical test parameters"
    },
    {
      "metric": "Provenance Specification Level (PSL)",
      "score": 35,
      "comment": "Lacks specification of data source context (blocks, prime mappings), timeline (experiment phases), and ra

In [20]:
# Extract and convert the revised pipeline into string format

import re

raw = response_h.content[0].text

json_txt = re.sub(r"^```json|```$", "", raw.strip(), flags=re.MULTILINE).strip()

review_obj = json.loads(json_txt)

revised_pipe = review_obj["revised_pipeline"]

revised_pipe_str = "\n".join(
    f"{step}" for idx, step in enumerate(revised_pipe)                     #{idx+1}. {step} if steps don't print as numbered list
)

print(revised_pipe_str)

1. import_libraries: Load required packages with library(readr), library(dplyr), library(broom), and library(effsize).
2. read_data: Read the dataset from CSV file using read.csv('data.csv') and verify n=21 participants with length(unique(data$Participant)).
3. filter_experimental_blocks: Filter dataset to include only Blocks 3 and 4 using filter(data, Block_Number %in% c(3,4)) to match the experimental design.
4. map_prime_conditions: Create cue condition variables by mapping Prime values to predictive conditions - identify which prime (2,4,8) serves as switch-predictive vs non-predictive for each participant using the existing swt_2, swt_8, stay_2, stay_4 columns.
5. isolate_switch_trials: Filter for switch trials only using filter(data, TrialType == 2).
6. separate_cue_conditions: Create two subsets from switch trials - predictive_switch using rows where the switch-predictive prime occurred, and nonpredictive_switch using rows where non-predictive prime occurred before switch trials

Or you can add a prompt here for either letting GPT make the suggested changes by you and anthropic or the user can make changes on their own. 


In [43]:
# ---------------------------------------------------------------
# 0   Helper: function-calling schema
# ---------------------------------------------------------------
tools_setup = [
    {
        "type": "function",
        "function": {
            "name": "submit_setup_plan",
            "description": "Return library list, dataset understanding, and cleaning steps",
            "parameters": {
                "type": "object",
                "properties": {
                    "libraries": {
                        "type": "array",
                        "description": "Each entry: package name and a one-line purpose",
                        "items": {
                            "type": "object",
                            "properties": {
                                "package": {"type": "string"},
                                "purpose": {"type": "string"}
                            },
                            "required": ["package", "purpose"]
                        }
                    },
                    "dataset_overview": {
                        "type": "array",
                        "description": "Schema summary (col name, inferred type, comment)",
                        "items": {
                            "type": "object",
                            "properties": {
                                "column":  {"type": "string"},
                                "type":    {"type": "string"},
                                "comment": {"type": "string"}
                            },
                            "required": ["column", "type", "comment"]
                        }
                    },
                    "final_pipeline": {
                        "type": "array",
                        "description": "Numbered list of pipeline steps (strings)",
                        "items": {"type": "string"}
                    },
                    "patch_applied": {
                        "type": "object",
                        "description": "Echo back exactly what USER_EDITS changed",
                        "properties": {
                            "steps_removed": {"type": "array", "items": {"type": "string"}},
                            "steps_added":   {"type": "array", "items": {"type": "string"}},
                            "steps_modified": {"type": "array", "items": {"type": "string"}}
                        },
                        "required": []
                    },
                    ###### cleaning part might be removed if considering that the data to be provided is a snippet##########
                    "cleaning_pipeline": {
                        "type": "array",
                        "description": "Numbered steps needed before analysis, or empty if none",
                        "items": {"type": "string"}
                    },
                    "needs_cleaning": {
                        "type": "boolean",
                        "description": "True if any cleaning is required"
                    }
                },
                "required": ["libraries", "dataset_overview", "cleaning_pipeline", "needs_cleaning"]
            }
        }
    }
]

# ---------------------------------------------------------------
# 1   Developer guard-rail for this turn
# ---------------------------------------------------------------
third_dev = (
    "You are now preparing to write R code for the FINAL analysis pipeline.\n"
    "- Ignore any other pipeline apart from the one provided in this message. \n"
    "- Task today: **plan**, do not write code.\n"
    "- Ignore any pipeline except the one shown above USER_EDITS.\n"
    "- Apply ONLY the edits listed in USER_EDITS. If USER_EDITS is empty/absent, "
    "make **no changes**.\n"
    "- **Return `final_pipeline` as the COMPLETE numbered list after edits; "
    "if no edits, return the original list unchanged.**\n"
    "- Record what changed in `patch_applied` (steps_added / steps_removed / steps_modified).\n"
    "- Output via the `submit_setup_plan` function only; no prose outside JSON.\n"
    "- List only R packages actually needed for the pipeline (e.g., readr, dplyr, stats).\n"
    "- For each package give one short clause on what it will do.\n"
    "- Infer dataset schema from the preview; reference columns exactly as named.\n"
    "- Decide whether cleaning/wrangling is required; if yes, give a numbered pipeline "
    "(e.g., convert factors, handle NAs). If no, explain via `needs_cleaning=false` and "
    "leave `cleaning_pipeline` empty."
)

# ---------------------------------------------------------------
# 2   User instruction for turn 3
# ---------------------------------------------------------------
third_user = (
    f"Here is the **first draft of analysis pipeline** that is approved:\n{revised_pipe_str}\n\n"
##### Add here any changes or addition to the pipeline that the model should consider

#   "No updates to the revised pipeline are needed at this time.\n\n"
######### OR ###########
   "### USER_EDITS \n"
   "Consider following updates to the pipeline:\n"
   "1. In step 2, after reading in the data there is no need to confirm the number of participants. It is 21. \n"
   "2. Add a step before step 5 to calculate the RT for all switch trials and all repeat trials and their accuracy rates. As mentioned in the article reference TARGET_RESULTS.\n"
   "Consider removing the following steps from the pipeline:\n"
   "1. Step 3 can be removed, as we only have 3 & 4 in the block_number.\n"
    "### END_EDITS\n\n"

#    "Dataset preview:\n"
#    f"{dataset}\n\n"
    "Please analyse the pipeline and dataset provided earlier as instructed."
)

# ---------------------------------------------------------------
# 3   Append to running history & call o3-mini
# ---------------------------------------------------------------
messages_client.extend([
    {"role": "developer", "content": third_dev},
    {"role": "user",      "content": third_user}
])



Can add a seperate prompt for the whole revised pipeline to be sure what exact changes are made.


In [46]:
response_2 = client.chat.completions.create(
    model     = "o3-mini",
    messages  = messages_client,
    tools     = tools_setup,
)

In [47]:
assistant_reply_2 = response_2.choices[0].message.tool_calls[0]
tool_call_args_2 = json.loads(assistant_reply_2.function.arguments)

# ------------------------------------------------------------------
#  Print for your own review 
# ------------------------------------------------------------------

libs_block = "\n".join(
    f"- {item['package']}: {item['purpose']}"
    for item in tool_call_args_2["libraries"]
)

schema_block = "\n".join(
    f"- {col['column']} ({col['type']}): {col['comment']}"
    for col in tool_call_args_2["dataset_overview"]
)

pipeline_fin = tool_call_args_2["final_pipeline"]
final_pipeline_block = "\n".join(pipeline_fin)                  # can add a error handle in case the pipeline is empty i.e. model schema fails
                                                                # if or statement to revert to revised_pipe OR add a value error

patch = tool_call_args_2.get("patch_applied", {})               # may be empty if no edits were made
added_block = "\n".join(f" -{s}" for s in patch.get("steps_added", [])) or " none (no edits)"
removed_block = "\n".join(f" -{s}" for s in patch.get("steps_removed", [])) or " none (no edits)"
modified_block = "\n".join(f" -{s}" for s in patch.get("steps_modified", [])) or " none (no edits)"

if tool_call_args_2["needs_cleaning"]:
    clean_steps = [s for s in tool_call_args_2["cleaning_pipeline"] if s.strip()]
    cleaning_block = "\n".join(
        f"{step}"
        for i, step in enumerate(clean_steps, start=1)
    )
    cleaning_block = "Cleaning pipeline:\n" + cleaning_block
else:
    cleaning_block = "Cleaning pipeline: none (data ready)"

summary_str = (
    "### Libraries\n"
    f"{libs_block}\n\n"
    "### Dataset overview\n"
    f"{schema_block}\n\n"
    f"{cleaning_block}\n\n"
    f"Cleaning required: {'yes' if tool_call_args_2['needs_cleaning'] else 'no'}\n\n"
    "### Final pipeline\n"
    f"{final_pipeline_block}\n\n"
    "### Edits applied\n"
    f"Steps added:\n{added_block}\n\n"
    f"Steps removed:\n{removed_block}\n\n"
    f"Steps modified:\n{modified_block}\n\n"
)

print(summary_str)

# ------------------------------------------------------------------
#  Append assistant-tool message to the running history
# ------------------------------------------------------------------

assistant_tool_msg_2 = {
    "role": "assistant",
    "content": None,                
    "tool_calls": [
        {
            "id":        assistant_reply_2.id,
            "type":      "function",
            "function": {
                "name":       assistant_reply_2.function.name,
                "arguments":  assistant_reply_2.function.arguments
            }
        }
    ]
}

    

messages_client.extend([
    assistant_tool_msg_2,
    { "role": "tool",
      "tool_call_id": assistant_reply_2.id,
      "content": "OK"
    },
    { "role": "assistant",                # recap message
      "name": "pipeline_v2",
      "content": "FINAL PIPELINE AFTER EDITS:\n" + final_pipeline_block
    }
])


### Libraries
- readr: Read CSV files into R.
- dplyr: Manipulate and filter data frames.
- broom: Tidy model outputs from statistical tests.
- stats: Perform statistical tests such as t.test and shapiro.test.
- effsize: Compute effect size measures like Cohen's d.

### Dataset overview
- Block_Number (numeric): Block identifier ranging from 3 to 4.
- Event_Number (numeric): Unique event identifier per trial.
- Prime (integer): Cue prime value, e.g., 2, 4, 8.
- PrimeVisible (numeric): Visibility flag for the prime.
- TaskType (numeric): Task type indicator.
- TrialType (numeric): Indicates repeat (0) or switch (2) trials.
- CorrResp (numeric): Correct response indicator.
- RT (numeric): Reaction Time in milliseconds.
- RespCorr (numeric): Accuracy flag for responses.
- lnum (numeric): Left number identifier.
- rnum (numeric): Right number identifier.
- lFont (numeric): Indicator if left number font is smaller.
- swt (numeric): Reaction time measure for switch trials.
- stay (numeric): 

In [48]:
# ------------------------------------------------------------------
# 0   Function schema: model asks clarifications or confirms ready
# ------------------------------------------------------------------
tools_clarify = [
    {
        "type": "function",
        "function": {
            "name": "ask_or_confirm",
            "description": (
                "If you need clarifications about dataset assumptions or logical gaps, "
                "list them. Otherwise confirm you are ready to code."
                "Also echo back any library/dataset edits requested by the user."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "needs_clarification": {
                        "type": "boolean",
                        "description": "True => you still have questions"
                    },
                    "questions": {
                        "type": "array",
                        "description": "List of clarification questions (ignored if flag is false)",
                        "items": {"type": "string"}
                    },
                    "ready_message": {
                        "type": "string",
                        "description": "Short confirmation like 'All clear, ready to code'; empty if questions exist"
                    },
                    "patch_mentioned":{
                        "type": "boolean",
                        "description": "True => user edits were mentioned in the USER_EDITS block; empty if no edits were requested"
                    },
                    "patch": {                                    
                        "type": "object",
                        "description": "Edits explicitly requested by user",
                        "properties": {
                            "libraries_add":  {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "R packages to ADD"
                            },
                            "libraries_drop": {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "R packages to REMOVE"
                            },
                            "dataset_notes":  {
                                "type": "string",
                                "description": "Update to dataset_overview (one short paragraph)"
                            }
                        },
                        "required": []
                    }
                },
                "required": ["needs_clarification", "questions", "ready_message", "patch"]
            }
        }
    }
]

# ------------------------------------------------------------------
# 1   Developer guard-rail for this turn
# ------------------------------------------------------------------
clarify_dev = (
    "We have finalised the analysis pipeline and selected libraries.\n"
    "User may ask for edits to libraries or dataset notes.\n"
    "- Apply ONLY the edits listed in the USER_EDITS block below (no new changes).\n"
    "- Do NOT write any R code yet.\n"
    "- Think about dataset assumptions and logical gaps.\n"
    "- If you need additional info, call `ask_or_confirm` with "
    "needs_clarification=true and a list of concise questions.\n"
    "- If everything is clear, call `ask_or_confirm` with "
    "needs_clarification=false and a short ready_message.\n"
    "- Output nothing outside the tool call."
)

# ------------------------------------------------------------------
# 2   User prompt
# ------------------------------------------------------------------
clarify_user = (
    "Refer to the pipeline in the message named 'pipeline_v2' for all steps.\n"
    "I agree with the final pipeline and the library list.\n"
    "Before you start coding, ask me any clarifications you need about:\n"
    " - Your assumptions on dataset cleaning/wrangling\n"
    " - Any logical or reasoning gaps you perceive\n"
    "If no clarifications are needed, just confirm you are ready."
##### Additional prompts to consider for any changes that need to be suggested in model understanding. #####
   "### USER_EDITS \n"
#   "Add libraries: \n"
#   "Delete libraries: \n"
#   "DATASET NOTES: \n"
########### OR if no change add none ############
    "No edits to the libraries or dataset notes are needed at this time.\n"
   "### END_EDITS \n"
)

# ------------------------------------------------------------------
# 3   Extend history & call o3-mini
# ------------------------------------------------------------------
messages_client.extend([
    {"role": "developer", "content": clarify_dev},
    {"role": "user",      "content": clarify_user}
])


In [49]:
response_3 = client.chat.completions.create(
    model    = "o3-mini",
    messages = messages_client,
    tools    = tools_clarify
)


In [50]:
assistant_reply_3 = response_3.choices[0].message.tool_calls[0]
tool_call_args_3 = json.loads(assistant_reply_3.function.arguments)

# ------------------------------------------------------------------
#  Print for your own review 
# ------------------------------------------------------------------

patch_flag = tool_call_args_3.get("patch_mentioned", False)

if patch_flag:
    p = tool_call_args_3["patch"]
    add_block  = "\n".join(f"- {lib}" for lib in p["libraries_add"])  or " none "
    drop_block = "\n".join(f"- {lib}" for lib in p["libraries_drop"]) or " none "
    dataset_notes_block = p["dataset_notes"].strip() or " none "
    patch_str = (
        "### User edits\n"
        f"**Libraries to add**:\n{add_block}\n\n"
        f"**Libraries to drop**:\n{drop_block}\n\n"
        f"**Dataset notes**:\n{dataset_notes_block}\n\n"
    )
else:
    patch_str = "### User edits\nNone – no changes requested\n\n"

if tool_call_args_3["needs_clarification"]:
    questions_block = "\n".join(f"- {q}" for q in tool_call_args_3["questions"])
    clarify_str = (
        "### Clarifications needed\n"
        f"{questions_block}\n"
    )
else:
    clarify_str = (
        "### Clarifications needed\n"
        "None – all clear \n\n"
        f"Ready message: {tool_call_args_3['ready_message']}"
    )

print(f"{patch_str}")
print(clarify_str)

# ------------------------------------------------------------------
#  Append assistant-tool message to the running history
# ------------------------------------------------------------------

assistant_tool_msg_3 = {
    "role": "assistant",
    "content": None,                
    "tool_calls": [
        {
            "id":        assistant_reply_3.id,
            "type":      "function",
            "function": {
                "name":       assistant_reply_3.function.name,
                "arguments":  assistant_reply_3.function.arguments
            }
        }
    ]
}


tool_response_msg_3 = {
    "role": "tool",
    "tool_call_id": assistant_reply_3.id,
    "content": "OK",
}

messages_client.extend([assistant_tool_msg_3, tool_response_msg_3])

### User edits
None – no changes requested


### Clarifications needed
None – all clear 

Ready message: All clear, ready to code.


In [51]:
# ------------------------------------------------------------------
# 1.  Developer guard-rail – force code-only output
# ------------------------------------------------------------------
code_dev = (
    "You have all clarifications.  Produce the FINAL R script.\n"
    "Output rules:\n"
    " - Return exactly ONE fenced code block: ```r ... ```\n"
    " - Begin with necessary `library()` calls.\n"
    " - Inline comments (#) must map to the numbered pipeline steps.\n"
    " - Use only variables present in the dataset header.\n"
    " - Saving output:\n"
    "     – If the user has asked to save a file, include the write step\n"
    "       (e.g., `write.csv()` or `saveRDS()`).\n"
    "     – Otherwise, print to console.\n"
    " - Plotting:\n"
    "     - You may generate a plot **only** if the article reference reports a figure\n"
    "       as part of the requested results.\n"
    "     – Use ggplot2; save with `ggsave()` if the user asked to save files.\n"
    " - Do NOT write prose outside the code block.\n"
)

# ------------------------------------------------------------------
# 2.  User message – your clarifications
#     (Replace the numbered answers with actual Q&A if different)
# ------------------------------------------------------------------
code_user = (
    #"Clarification answers:\n"
    #"1. Yes the exclusion criteria metnioned by you is correct. \n"
    #"2. No other cleaning steps are required.\n"

    "No clarification was required by you.\n\n"

    #"Additional instructions:\n"
    ############# Instruction to save results csv ###############
    #" - Please **save** the final summary table as 'summary_table.csv'.\n"
    ############# Instrctions to create and save plots ################
    #" - Also recreate the plot similar to the one shared earlier. Create it using the dataset provided,\n"
    #"  and save it as 'plot.png'.\n"
    ####################################################################
    "Now produce the code."
)

# ------------------------------------------------------------------
# 3.  Append to chat history and call the model
# ------------------------------------------------------------------
messages_client.extend([
    {"role": "developer", "content": code_dev},
    {"role": "user",      "content": code_user}
])



In [52]:
response_4 = client.chat.completions.create(
    model="o3-mini",
    messages=messages_client,
)
print("Response for the fourth prompt:\n")
print(response_4.choices[0].message.content)
assistant_reply_4 = response_4.choices[0].message.content

Response for the fourth prompt:

```r
# 1. import_libraries: Load required packages using readr, dplyr, broom, stats, and effsize.
library(readr)   # For reading CSV files
library(dplyr)   # For data manipulation
library(broom)   # For tidying statistical test outputs
library(stats)   # For t.test and shapiro.test
library(effsize) # For computing Cohen's d

# 2. read_data: Read the dataset from CSV file; participant count check is skipped.
data <- read.csv("data.csv")

# 3. map_prime_conditions: Create cue condition variable mapping predictive conditions for switch trials.
data <- data %>%
  mutate(CueCondition = case_when(
    TrialType == 2 & swt_8.1 == 1 ~ "predictive",
    TrialType == 2 & swt_2.1 == 1 ~ "nonpredictive",
    TRUE ~ NA_character_
  ))

# 4. calculate_overall_descriptive: Calculate overall descriptive measures for switch and repeat trials.
# For switch trials (TrialType == 2)
overall_switch <- data %>%
  filter(TrialType == 2) %>%
  group_by(Participant) %>%
  summar

In [ ]:
################ ANTHRPOIC USED CONTEXTUALLY WE ARE NOT CONSIDERING THE LAST PROMPT ONLY THE REPLY ####################

In [53]:
# ------------------------------------------------------------------
# 1  System prompt  – guard-rails + what counts as a “fault”
# ------------------------------------------------------------------
system_prompt_1 = f"""
You are a senior data-science reviewer. Checking R code based on below metrics, considering the pipeline provided by the coder.

Allowed scope
-------------
- Logical correctness (does each step implement the stated pipeline?)
- function compatibility with the libraries used (e.g. does the function exist, is it used correctly)  
- Data-safety issues (e.g. NA handling if NAs possible, factor coercion, wrong column names)  
- Runtime efficiency (vectorised ops, avoiding unnecessary copies)  
- Ignore spelling, commenting style, or aesthetic refactors  
- Do not add new tests, plots, or validations unless the **pipeline explicitly requires them**.

Fault vs. nit-pick
------------------
A *fault* is any construct that could give wrong numbers, crash, or be slow for the stated dataset.
Examples to flag:
- Forgetting `na.rm = TRUE` when summarising numeric data that may contain NAs  
- Using a function that does not exist in the loaded libraries
- Referencing a column not in the dataset  
- Double-reading the same file or using a redundant loop

Output format (MUST follow exactly)
-----------------------------------
### Feedback
- Metric: <name> — <concise remark>  (only if a real fault or big efficiency gain)

### Revised_Code
```r
# complete runnable script in R
```
### Change_Log
 - <what you changed & why> (reference a Feedback bullet)
"""

# ------------------------------------------------------------------
# 2  User prompt – supply code to review
# ------------------------------------------------------------------

user_prompt_1 = (f"""
Here is the peer-reviewer's R code:
{assistant_reply_4}
Remember: comment only on logic/runtime faults or big efficiency wins.
Do not invent extra analyses, plots, or style tweaks.
"""
#"""
#While reviewing the code, consider the following updates to the pipeline:

#### USER_UPDATES 
#Make below changes to the pipeline and consider the changes in your review:
#No changes are needed.
#### END_USER_UPDATES
#"""
)

messages_checker = ([{"role": "assistant", "content": f"{final_pipeline_block}"},
    {"role": "user", "content": f"{user_prompt_1}"}])

In [54]:
messages_checker

[{'role': 'assistant',
  'content': "1. import_libraries: Load required packages using library(readr), library(dplyr), library(broom), library(stats), and library(effsize).\n2. read_data: Read the dataset from CSV file using read.csv('data.csv'); no participant number check is performed as it is predetermined to be 21.\n3. map_prime_conditions: Create cue condition variables by mapping the Prime column values to predictive conditions using the swt_2, swt_8, stay_2, and stay_4 columns.\n4. calculate_overall_descriptive: Calculate overall descriptive measures for switch trials and repeat trials by grouping by Participant and computing mean RT using mean(RT, na.rm=TRUE) and accuracy as mean(CorrResp).\n5. isolate_switch_trials: Filter for switch trials only using filter(data, TrialType == 2).\n6. separate_cue_conditions: Create two subsets from switch trials - one for predictive switch trials using filter(., swt_8.1 == 1) and one for nonpredictive switch trials using filter(., swt_2.1 == 

In [55]:
response_h_1 = checker.messages.create(
    model = 'claude-sonnet-4-20250514',
    system = f"{system_prompt_1}",
    messages = messages_checker,
    max_tokens = 5000
)



In [56]:
print(response_h_1.content[0].text)
checker_code = response_h_1.content[0].text

### Feedback
- Metric: Logical correctness — Step 4 calculates overall descriptives but doesn't actually compute overall switch cost as required by step 8; the switch cost calculation is done separately but uses different aggregation logic
- Metric: Data-safety issues — Missing validation that required columns (TrialType, swt_8.1, swt_2.1, RT, CorrResp, Participant) exist in the dataset before processing
- Metric: Runtime efficiency — Using `merge()` instead of more efficient `full_join()` from dplyr, and creating unnecessary intermediate datasets

### Revised_Code
```r
# 1. import_libraries: Load required packages using readr, dplyr, broom, stats, and effsize.
library(readr)   # For reading CSV files
library(dplyr)   # For data manipulation
library(broom)   # For tidying statistical test outputs
library(stats)   # For t.test and shapiro.test
library(effsize) # For computing Cohen's d

# 2. read_data: Read the dataset from CSV file; participant count check is skipped.
data <- read.csv(

We can try to use anthropic to adjust the final code rather than running it again through gpt.